# 20. 向量化与广播

<!-- module-learning-arc:start -->
> **NumPy 模块主线｜第 5 / 6 步：用广播替代重复循环**
>
> **持续应用背景：** 为区域仓库建立补货预警矩阵：把门店、商品、库存和需求组织成数组，逐步完成定位、广播计算、排序和抽样复核。
>
> **承接上一阶段：** 形状、合并与拆分  →  **本章任务：** 向量化与广播  →  **下一步：** 统计计算与随机抽样
>
> **大作业连接：** 本章练习将成为《连锁门店补货预警矩阵》的一部分，最终需要把数组建模、风险筛选、广播计算和抽样复核组合成一份可执行的补货清单。
<!-- module-learning-arc:end -->


## 本章场景

真实计算里，数据常常是一整个数组一起处理——比如一批商品统一加价、一周的销售额整体调整。



## 本章目标

学完本章，你将能够：

- **理解**：理解向量化运算与广播规则。
- **操作**：能对数组整体做运算（按列/标量广播）。
- **迁移**：能用向量化运算批量计算（如价格×数量）避免写循环。


## 20.1 核心概念

**背景引入**：真实计算里，数据常常是一整个数组一起处理——比如一批商品统一加价、一周的销售额整体调整。用 for 循环逐个取数不仅慢，写起来也啰嗦，还容易踩到很多隐藏的坑。向量化让 NumPy 在底层一次性处理整批数据，广播则让不同形状的数组也能按规则自然对齐运算。学完这一章，你就能用短短几行写出又快又清晰的批量计算。

- 向量化把运算交给NumPy底层实现。
- 广播从末尾维度开始比较，维度相等或其中一个为1时兼容。
- 能广播不代表业务含义正确，仍需检查轴和单位。

**口诀**：末尾对齐才广播，有个维度是 1 就能扩；能算不等于算对，先确认轴和单位。


## 20.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 向量化加减 | `np.array()` | 数组运算会逐元素执行，不需要手写for循环。 | 广播成功但沿错了业务维度 |
| np.sqrt() | `np.array()`、`np.sqrt()` | NumPy通用函数可以批量处理整个数组。 | 用Python循环重复NumPy原生操作 |
| 广播 | `np.array()` | 一维数组可以沿兼容的维度与二维数组运算。 | 数组形状不兼容时盲目reshape |
| np.where() 向量化条件 | `np.array()`、`np.where()` | where可以一次性给整列数值打标签。 | 广播成功但沿错了业务维度 |
| np.clip() | `np.array()`、`np.clip()` | clip把过小或过大的值限制在指定区间内。 | 用Python循环重复NumPy原生操作 |


## 20.3 示例 1：向量化计算

数组运算会逐元素执行，不需要手写循环。

**背景引入**：一批商品各有单价和销量，要算每笔金额，再对满 300 元的订单统一打九折。用 for 循环逐个乘、逐个判断，代码又长又慢；向量化让乘法和条件判断对整批数据一次算完。

**讲解**：数组与数组相乘是逐元素对齐计算（prices × quantities），np.where 按条件整批挑选结果，等于把 if/else 循环写在同一行。

- amounts = prices * quantities：同长度数组对应位置相乘，一次算完所有金额；
- np.where(amounts >= 300, amounts * 0.9, amounts)：满足条件的走 9 折，否则保留原值，整批打标；
- 全程不写 for 循环，NumPy 在底层批量处理，快又简洁；
- **口诀**：整批相乘用向量，条件整批交给 where，一处循环全免写。


<!-- math-foundation:chapter-20 -->
### 数学推导｜广播把一维规则应用到矩阵

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜把向量看成一行。** 长度为 $n$ 的 $b$ 可解释为 $1\times n$ 数组。

**第 2 步｜沿缺少的行轴复制。** 广播得到的概念矩阵满足

$$
B_{ij}=b_j,\qquad i=1,\ldots,m
$$

**第 3 步｜执行逐位置运算。** 因此 $C=A+B$ 等价于对每一行应用同一组列规则：$C_{ij}=A_{ij}+b_j$。NumPy 通常不会真的复制整张 $B$，但语义与此相同。

**把上面的关系收束为本章计算式：**

$$
C_{ij}=A_{ij}+b_j
$$

**符号解释：** $A$ 是 $m\times n$ 矩阵，$b$ 长度为 $n$，同一个 $b_j$ 会作用于每一行。

**代码对应：** `C = A + b`；运行前用 `A.shape` 与 `b.shape` 确认 $b$ 对应哪条轴。

**使用边界：** 能够广播不等于业务轴正确；门店规则和商品规则放错轴仍会得到无报错的错误结果。


In [ ]:
import numpy as np

prices = np.array([128.0, 299.0, 59.0, 499.0])
quantities = np.array([2, 1, 3, 1])
amounts = prices * quantities
discounted = np.where(amounts >= 300, amounts * 0.9, amounts)
print(amounts)
print(discounted)


## 20.4 示例 2：通用函数

ufunc支持批量数学运算和返回多个数组。

**背景引入**：分析时经常要对整个数组做“开平方”“取对数”这类数学运算，偶尔还要把小数拆成整数部分和小数部分分别用。一个个算太笨，通用函数（ufunc）能对整批数一次算完。

**讲解**：np.sqrt、np.log1p 是通用函数，对数组里每个元素批量处理；np.modf 还能同时返回“小数部分、整数部分”两个数组。

- np.sqrt(values)、np.log1p(values)：对整批数字分别开方、取 log(x+1)，逐个元素计算；
- fraction, integer = np.modf(arr)：一次得到小数部分和整数部分两个数组，一一对应；
- 通用函数写法短、速度快，比自己写循环更稳；
- **口诀**：数学运算找 ufunc、一次算整批；要拆整数和小数，modf 一手抓两个。


In [ ]:
values = np.array([1, 4, 9, 16])
print("平方根:", np.sqrt(values))
print("对数:", np.log1p(values))
fraction, integer = np.modf(np.array([1.25, 2.8, 3.0]))
print("小数部分:", fraction)
print("整数部分:", integer)


## 20.5 示例 3：二维广播

行向量可以与二维矩阵的每一行进行运算。

**背景引入**：不同区域的销售矩阵按月列着，要对每个月统一乘“季节系数”，还要算每个区域相对基数的指数。一个形状是 2×3、一个是长度 3，形状不同却要运算；广播就是让它们自动对齐、省去手动复制。

**讲解**：一维因子与二维矩阵运算时，广播会沿“列”维度把因子复制到每一行；再配合一个“列向量”当基数，就能算出每个区域的相对指数。

- sales * monthly_factor：月因子长度 3 与 3 列对齐，对每一行都乘上对应月份系数；
- sales / region_base * 100：region_base 是 (2,1) 列向量，按“行”对齐到每个区域基数，算出区域指数；
- 广播从末尾维度开始比较，维度相等或其中一个为 1 就能兼容；
- **口诀**：末尾对齐才广播，有个维度是 1 就能扩；能算不等于算对，先确认轴和单位。


In [ ]:
sales = np.array([[120, 150, 180], [90, 130, 160]])
monthly_factor = np.array([1.0, 1.05, 1.1])
adjusted = sales * monthly_factor
region_base = np.array([[100], [80]])
index_values = sales / region_base * 100
print(adjusted)
print(index_values)


## 20.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# 向量化加减
# 数组运算会逐元素执行，不需要手写for循环。
import numpy as np

sales = np.array([100, 120, 150])
adjusted = sales * 1.1
print(adjusted)


In [ ]:
# np.sqrt()
# NumPy通用函数可以批量处理整个数组。
import numpy as np

values = np.array([1, 4, 9, 16])
print(np.sqrt(values))


In [ ]:
# 广播
# 一维数组可以沿兼容的维度与二维数组运算。
import numpy as np

sales = np.array([[100, 120, 140], [80, 90, 110]])
factor = np.array([1.0, 1.05, 1.1])
print(sales * factor)


In [ ]:
# np.where() 向量化条件
# where可以一次性给整列数值打标签。
import numpy as np

amount = np.array([120, 580, 1280])
labels = np.where(amount >= 500, "重点", "普通")
print(labels)


In [ ]:
# np.clip()
# clip把过小或过大的值限制在指定区间内。
import numpy as np

scores = np.array([-5, 65, 108])
print(np.clip(scores, 0, 100))


**练一练 17.6**：围绕本章的核心操作设计一个小任务。先创建价格数组 price；① 用一次向量化乘法把所有价格提高 20%，保存为 price_after；② 把一个形状为 (3,) 的调整因子 factor 沿列广播到形状 (2,3) 的销售额 sales 上，保存为 adjusted；③ 用 np.clip 把金额数组 amount 限制在 [0, 100]，再用 np.where 给达标（≥100）的项打上「达标」标签，保存为 tag。数据用简单的数值数组即可。


In [ ]:
# 请在下方填写代码
import numpy as np

price = np.array([10, 20, 30])

# TODO ①: 用一次向量化乘法把 price 整体提高 20%，保存在 price_after（不要写 for 循环）

# TODO ②: 让形状为 (3,) 的 factor 沿列广播到 sales 上，保存在 adjusted
sales = np.array([[100, 120, 140], [80, 90, 110]])
factor = np.array([1.0, 1.05, 1.1])

# TODO ③: 用 np.clip 把 amount 限制在 [0, 100]，再用 np.where 给 >=100
# 的打上「达标」标签，分别保存在 clipped 和 tag
amount = np.array([60, 120, 200])


In [ ]:
import numpy as np

price = np.array([10, 20, 30])

# ① 向量化乘法：整个数组一次性乘 1.2，无需 for 循环
price_after = price * 1.2

# ② 广播：形状 (3,) 的 factor 沿列方向广播到形状 (2,3) 的 sales
sales = np.array([[100, 120, 140], [80, 90, 110]])
factor = np.array([1.0, 1.05, 1.1])
adjusted = sales * factor

# ③ clip 限制在 [0,100]，where 按条件打标签
amount = np.array([60, 120, 200])
clipped = np.clip(amount, 0, 100)
tag = np.where(amount >= 100, "达标", "未达标")

print(price_after)
print(adjusted.shape, adjusted[:, 0])
print(clipped, tag)


**输出解读**：① `price_after = price * 1.2` 一次对所有价格上浮 20%，无需循环；② 长度 3 的 `factor` 沿列广播到 `(2,3)` 的 `sales`，`adjusted` 形状仍为 `(2,3)`；③ `np.clip(amount, 0, 100)` 把超出区间的值截到边界，`np.where(amount >= 100, "达标", "未达标")` 整批打标签。要记住：广播"算得动"不等于"算得对"，先想清轴和单位。


## 20.7 独立迁移练习

先预测 shape，再修改一个数组或筛选条件，解释结果变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 20.8 本章实训：axis与布尔筛选

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np

matrix = np.arange(1, 13).reshape(3, 4)
print("原数组：\n", matrix)
print("每行合计：", matrix.sum(axis=1))
print("每列合计：", matrix.sum(axis=0))


### 20.8.1 第一个结果怎么读

`axis=1` 保留行，沿列方向计算；`axis=0` 保留列，沿行方向计算。先看 shape，再解释结果长度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
even = matrix[matrix % 2 == 0]
print("偶数：", even)
print("偶数数量：", even.size)
print("偶数平均值：", even.mean())


### 20.8.2 第二个结果怎么读

第二个实验不改原数组，而是用布尔条件筛选新数组。请思考：如果条件改成 `matrix > 8`，输出会怎样变化？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 20.9 错误恢复：数组形状不匹配怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import numpy as np

matrix = np.arange(6).reshape(2, 3)
try:
    result = matrix + np.array([10, 20])
except ValueError as error:
    print("形状问题：", type(error).__name__)
    result = matrix + np.array([10, 20, 30])
print("修复后的结果：")
print(result)


### 20.9.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

先看两个数组的 shape，再判断能否广播。修复不是随意 reshape，而是让数据结构和业务含义一致。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 20.10 易错点提醒

- 广播成功但沿错了业务维度
- 用Python循环重复NumPy原生操作
- 数组形状不兼容时盲目reshape


## 20.11 练习与作业

1. 建立3个商品×4个月销量矩阵
2. 用长度为4的季节系数调整
3. 输出调整后每个商品合计

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 20.12 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“建立3个商品×4个月销量矩阵”。
2. **独立完成**：不复制示例代码，完成“用长度为4的季节系数调整”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出调整后每个商品合计”，用一两句话说明你修改了什么。

### 20.12.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 20.12.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import numpy as np

# TODO: 使用广播进行季节调整
# TODO：请在下方完成 —— 17.12 练习与作业 1. 建立3个商品×4个月销量矩阵 2. 用长度为4的季节系数调整 3. 输出调整后每个商品合计


In [ ]:
import numpy as np

sales = np.array([[12, 15, 18, 20], [8, 11, 13, 16], [20, 22, 25, 28]])
season_factor = np.array([0.9, 1.0, 1.1, 1.2])
adjusted = sales * season_factor
print(adjusted)
print("商品合计:", adjusted.sum(axis=1))


## 20.13 小结

利用向量化和广播替代逐元素循环，写出简洁高效的数组计算。

**迁移思考**：

1. 如果需要用不同的季节系数调整每个地区的销售额（3个地区×4个月），系数数组应该是什么形状？
2. 为什么广播成功不代表业务含义正确？应该如何避免沿错误的轴广播？



### 20.13.1 你已经掌握

- 执行逐元素运算
- 使用通用函数
- 理解广播规则
- 诊断维度不兼容


### 20.13.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 20.13.3 需要注意

- 广播成功但沿错了业务维度
- 用Python循环重复NumPy原生操作
- 数组形状不兼容时盲目reshape


### 20.13.4 完成检查

- [ ] 能够执行逐元素运算
- [ ] 能够使用通用函数
- [ ] 能够理解广播规则
- [ ] 能够诊断维度不兼容


### 20.13.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
